# PubMed Oncology Qwen3.6 27B SFT Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** unsloth/Qwen3.6-27B-NVFP4

**Dataset:** PubMed oncology datagen JSONL — multi-turn QA, continuation, treatment reasoning, beyond-evidence, and self-correction conversations across 11 cancer types

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Chat Template:** `<|im_start|>role\ncontent<|im_end|>` (ChatML)

**Architecture:** This is Phase 1 (SFT). Teaches the model clinical oncology reasoning with thinking chains. Phase 2 (DPO) refines response quality using preference pairs.

## How to run
**Press "Run All" and walk away.** Every cell is self-contained and runs in order.

## 1. Setup — Configuration, Environment, GPU Check

Installs missing packages, verifies GPU, sets all paths and hyperparameters. Safe to re-run.

In [ ]:
import os, sys, subprocess, importlib, importlib.util

# =========================== SHARED HUGGING FACE CACHE ===========================
# Use the cache shared with vLLM. Must be set before importing Unsloth or Transformers.
for _cache_dir in ("/root/.cache/huggingface", "/home/spark/.cache/huggingface"):
    if os.path.isdir(_cache_dir):
        os.environ["HF_HUB_CACHE"] = _cache_dir
        break

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 0: NEUTER GIT                                                         ║
# ║                                                                              ║
# ║  gptqmodel's startup banner shells out to `git rev-parse` to stamp its      ║
# ║  version. /workspace/* is owned by the host user but git runs as root in    ║
# ║  this container → "dubious ownership" failure, which trips up downstream    ║
# ║  code that wasn't expecting a non-zero git exit. We don't use git for       ║
# ║  anything here, so prepend a fake `git` to PATH that always exits 0.        ║
# ║  Must run before any heavy imports (unsloth → transformers → gptqmodel).    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import tempfile, stat
_fake_git_dir = tempfile.mkdtemp(prefix="nogit-")
_fake_git = os.path.join(_fake_git_dir, "git")
with open(_fake_git, "w") as _f:
    _f.write("#!/bin/sh\nexit 0\n")
os.chmod(_fake_git, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
os.environ["PATH"] = _fake_git_dir + os.pathsep + os.environ.get("PATH", "")

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 1: INSTALL MISSING PACKAGES (safe to re-run, never clobbers NGC torch)║
# ║                                                                              ║
# ║  ORDER MATTERS:                                                              ║
# ║    1. torch check (no side effects)                                          ║
# ║    2. pip install small packages                                             ║
# ║    3. fix causal_conv1d (NGC ships broken build w/o CUDA extension)          ║
# ║    4. import unsloth FIRST (BEFORE transformers — required for patching)     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def _pip(*args, env_extra=None):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    return True

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# ── 1a. Verify NGC CUDA PyTorch is intact ──
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "+cpu" in torch.__version__ or "cpu" in torch.__version__:
        print("  NGC CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through. Check Portainer: runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")

print(f"  torch {torch.__version__} — CUDA {torch.version.cuda} — GPU: {torch.cuda.get_device_name(0)}")

# ── 1b. Small utility packages ──
for module, install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "matplotlib":  ["install", "-q", "matplotlib"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(module) is None:
        print(f"  Installing {install_args[-1]}...")
        _pip(*install_args)

# ── 1c. Match compressed-tensors to the NVFP4 checkpoint format ──
# This checkpoint declares compressed-tensors 0.17.2.a20260707 in config.json.
# Older releases mis-handle its overlapping mixed-precision groups during load.
import importlib.metadata

_COMPRESSED_TENSORS_VERSION = "0.17.2a20260707"
try:
    _installed_ct_version = importlib.metadata.version("compressed-tensors")
except importlib.metadata.PackageNotFoundError:
    _installed_ct_version = None

if _installed_ct_version != _COMPRESSED_TENSORS_VERSION:
    print(f"  Installing compressed-tensors {_COMPRESSED_TENSORS_VERSION} "
          f"(current: {_installed_ct_version or 'missing'})...")
    if not _pip("install", "-q", "--no-deps", "--force-reinstall",
                f"compressed-tensors=={_COMPRESSED_TENSORS_VERSION}"):
        raise RuntimeError("Failed to install the compressed-tensors version required by the NVFP4 checkpoint")
    for _k in list(sys.modules.keys()):
        if _k == "compressed_tensors" or _k.startswith("compressed_tensors."):
            del sys.modules[_k]
    importlib.invalidate_caches()

print(f"  compressed-tensors: {importlib.metadata.version('compressed-tensors')} (NVFP4 checkpoint match)")

# ── 1d. Fix causal_conv1d ──
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing — rebuilding from source (~3 min)...")
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
              "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext — uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed — uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# ── 1e. Import unsloth FIRST, then transformers ──
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
print(f"  transformers {transformers.__version__}")

# ── 1f. Report final state ──
print()
for name, mod in [("unsloth", "unsloth"), ("transformers", "transformers"), ("trl", "trl"),
                   ("causal_conv1d", "causal_conv1d")]:
    m = _check_import(mod)
    v = getattr(m, "__version__", "installed") if m else "n/a"
    status = "OK" if m else "FALLBACK" if name == "causal_conv1d" else "MISSING"
    print(f"  {name:25s} {v:20s} [{status}]")

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 2: CONFIGURATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print()
print("=" * 60)
print("CONFIGURATION")
print("=" * 60)

# --- Paths (auto-detect Docker vs host) ---
if os.path.exists("/workspace/training/pubmed"):
    PROJECT_ROOT = "/workspace/training/pubmed"
    _env = "Docker (Unsloth container)"
elif os.path.exists("/workspace/pubmed"):
    PROJECT_ROOT = "/workspace/pubmed"
    _env = "Docker (legacy mount)"
else:
    PROJECT_ROOT = "/home/spark/projects/training/pubmed"
    _env = "Host (VS Code / venv)"

DATA_DIR    = f"{PROJECT_ROOT}/data"
OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== MODEL CONFIGURATION ===========================
BASE_LLM        = "unsloth/Qwen3.6-27B-NVFP4"
MODEL_NAME_BASE = "pubmed_oncologist_v2_sft_qwen36_27b_unsloth_nvfp4"

# =========================== INPUT DATA ===========================
# Standard multi-turn ShareGPT SFT dataset
INPUT_DATA_FILE = f"{DATA_DIR}/training-data/pubmed_oncologist_v2/pubmed_oncologist_combined_sharegpt.jsonl"

# Cap on number of conversations used. 0 = use all loaded conversations.
# Applied AFTER loading the JSONL, BEFORE quality validation. Set e.g. 500
# for a smoke-test run, 3000 for a quick LoRA, leave 0 for full training.
SFT_MAX_EXAMPLES = 9000

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR     = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR     = f"{OUTPUT_BASE_DIR}/lora_adapters"

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH  = 4096
BATCH_SIZE      = 2
GRAD_ACCUM      = 4        # effective batch = 1 * 8 = 8
LEARNING_RATE   = 2e-4
TARGET_EPOCHS   = 1

# =========================== LoRA CONFIGURATION ===========================
LORA_R              = 32
LORA_ALPHA          = 32
LORA_DROPOUT        = 0
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# =========================== INFERENCE TEST ===========================
TEST_PROMPT = "A 58-year-old woman with BRCA1-mutated high-grade serous ovarian cancer has progressed after platinum-based chemotherapy and a PARP inhibitor. What are the next treatment options?"

# --- Print summary ---
print(f"  Environment:  {_env}")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  Base model:   {BASE_LLM}")
print(f"  Model name:   {MODEL_NAME_BASE}")
print(f"  Input data:   {INPUT_DATA_FILE}")
print(f"  SFT cap:      {SFT_MAX_EXAMPLES or 'ALL'}")
print(f"  LoRA output:  {LORA_OUTPUT_DIR}")
print(f"  LoRA:         r={LORA_R}, alpha={LORA_ALPHA}, targets={len(LORA_TARGET_MODULES)} modules")
print(f"  Training:     batch={BATCH_SIZE} x grad_accum={GRAD_ACCUM} = effective {BATCH_SIZE*GRAD_ACCUM}")
print(f"  Precision:    NVFP4 LoRA")

# --- Verify paths ---
for path, label in [(INPUT_DATA_FILE, "Training data"), (PROJECT_ROOT, "Project root")]:
    exists = os.path.exists(path)
    print(f"  {'OK' if exists else 'MISSING':7s} {label}: {path}")
    if not exists:
        raise FileNotFoundError(f"{label} not found: {path}")

print()
print("Setup complete. All cells below can run sequentially.")

ENVIRONMENT SETUP
  torch 2.10.0a0+b558c986e8.nv25.11 — CUDA 13.0 — GPU: NVIDIA GB10
  compressed-tensors: 0.17.2a20260707 (NVFP4 checkpoint match)
  causal_conv1d: OK (CUDA extension loaded)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  transformers 5.15.0.dev0

  unsloth                   2026.7.5             [OK]
  transformers              5.15.0.dev0          [OK]
  trl                       0.24.0               [OK]
  causal_conv1d             0.0.local            [OK]

CONFIGURATION
  Environment:  Docker (Unsloth container)
  PROJECT_ROOT: /workspace/training/pubmed
  Base model:   unsloth/Qwen3.6-27B-NVFP4
  Model name:   pubmed_oncologist_v2_sft_qwen36_27b_unsloth_nvfp4
  Input data:   /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2/pubmed_oncologist_combined_sharegpt.jsonl
  SFT cap:      3000
  LoRA output:  /workspace/training/pubmed/output/pubmed_oncologist_v2_

## 2. Load Dataset

Load the combined multi-turn ShareGPT JSONL from datagen.

- 33,349 conversations across 11 cancer types
- Data types: QA (with thinking), continuation, treatment reasoning, beyond-evidence, self-correction
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`
- System prompts are extracted from the JSONL at load time (stays in sync with datagen)

In [2]:
import json, os
from collections import defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING SHAREGPT SFT DATA")
print(f"  File: {INPUT_DATA_FILE}")

raw_rows = []
cancer_by_index = []
cancer_type_counts = defaultdict(int)
source_counts = defaultdict(int)

with open(INPUT_DATA_FILE) as f:
    for line in f:
        row = json.loads(line)

        cancer = row.get("cancer_type", "unknown")
        source = row.get("source", "unknown")
        cancer_type_counts[cancer] += 1
        source_counts[source] += 1

        # Accept native messages rows for compatibility.
        if "messages" in row:
            raw_rows.append({"messages": row["messages"]})
        elif "conversations" in row:
            # Load the standard ShareGPT conversations format.
            raw_rows.append({"conversations": row["conversations"]})
        else:
            raise ValueError("Row missing both 'messages' and 'conversations'")

        cancer_by_index.append(cancer)

dataset = HFDataset.from_list(raw_rows)

# Apply SFT cap from config — stratified per cancer type, deterministic seed.
if SFT_MAX_EXAMPLES:
    import random
    random.seed(42)
    if SFT_MAX_EXAMPLES < 0:
        raise ValueError(f"SFT_MAX_EXAMPLES must be 0 or positive, got {SFT_MAX_EXAMPLES}")
    if len(dataset) > SFT_MAX_EXAMPLES:
        cancers_sorted = sorted(cancer_type_counts.keys(), key=lambda c: -cancer_type_counts[c])
        n_types = len(cancers_sorted)
        base = SFT_MAX_EXAMPLES // n_types
        extras = SFT_MAX_EXAMPLES - base * n_types
        target = {c: base + (1 if i < extras else 0) for i, c in enumerate(cancers_sorted)}

        idx_by_cancer = defaultdict(list)
        for i, c in enumerate(cancer_by_index):
            idx_by_cancer[c].append(i)

        selected_idx = []
        per_type_taken = {}
        for c in cancers_sorted:
            pool = idx_by_cancer[c]
            random.shuffle(pool)
            take = min(len(pool), target[c])
            per_type_taken[c] = take
            selected_idx.extend(pool[:take])
        random.shuffle(selected_idx)

        print(f"  Stratified cap: target {SFT_MAX_EXAMPLES} across {n_types} cancer types "
              f"(~{base}/type, +1 for top {extras})")
        print(f"  Actually selected: {len(selected_idx)} rows")
        for c in cancers_sorted:
            short = "(undersized)" if per_type_taken[c] < target[c] else ""
            print(f"    {c:30s} {per_type_taken[c]:>5d} / {target[c]:<5d} {short}")

        dataset = dataset.select(selected_idx)
        cancer_type_counts = defaultdict(int, per_type_taken)
    else:
        print(f"  SFT_MAX_EXAMPLES={SFT_MAX_EXAMPLES}, but only {len(dataset)} examples available; using all")

print(f"\n{'='*50}")
print(f"Total dataset: {len(dataset)} conversations")
print(f"Cancer types: {len(cancer_type_counts)}")
print(f"Sources: {len(source_counts)}")
print(f"Columns: {dataset.column_names}")

print(f"\nPer-cancer breakdown:")
for ct, c in sorted(cancer_type_counts.items(), key=lambda x: -x[1]):
    print(f"  {ct:30s} {c:>5d} conversations")

print(f"\nPer-source breakdown:")
for src, c in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"  {src:30s} {c:>5d} rows")


LOADING SHAREGPT SFT DATA
  File: /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2/pubmed_oncologist_combined_sharegpt.jsonl
  Stratified cap: target 3000 across 11 cancer types (~272/type, +1 for top 8)
  Actually selected: 3000 rows
    multi                            273 / 273   
    pubmed_brain_tumour              273 / 273   
    pubmed_bone_cancer               273 / 273   
    pubmed_colon_cancer              273 / 273   
    pubmed_breast_cancer             273 / 273   
    pubmed_ovarian_cancer            273 / 273   
    pubmed_lung_cancer               273 / 273   
    pubmed_kidney_cancer             273 / 273   
    pubmed_gastric_cancer            272 / 272   
    pubmed_skin_cancer               272 / 272   
    pubmed_prostate_cancer           272 / 272   

Total dataset: 3000 conversations
Cancer types: 11
Sources: 1
Columns: ['conversations']

Per-cancer breakdown:
  multi                            273 conversations
  pubmed_brain_tumour          

## 3. Validate & Summarize Dataset

Verify data quality: turn structure, non-empty responses, thinking tag presence.

In [3]:
from collections import Counter

bad_examples = []
empty_final_responses = []
tool_call_examples = 0

def _get_messages(example):
    if "messages" in example:
        return example["messages"]
    if "conversations" in example:
        # Fallback support for ShareGPT rows.
        role_map = {"system": "system", "human": "user", "gpt": "assistant"}
        return [{"role": role_map[t["from"]], "content": t["value"]} for t in example["conversations"]]
    raise ValueError("Example missing both messages and conversations")

for i, example in enumerate(dataset):
    msgs = _get_messages(example)

    if len(msgs) < 3:
        bad_examples.append((i, f"Expected >=3 messages, got {len(msgs)}"))
        continue

    # Must start with system, then user for this training format.
    if msgs[0].get("role") != "system" or msgs[1].get("role") != "user":
        bad_examples.append((i, "Expected first two roles to be system,user"))
        continue

    # Final assistant content must be non-empty.
    last = msgs[-1]
    if last.get("role") != "assistant" or not str(last.get("content", "")).strip():
        empty_final_responses.append(i)

    # Confirm the selected standard SFT dataset contains no explicit tool calls.
    has_tool_call = any(m.get("role") == "assistant" and m.get("tool_calls") for m in msgs)
    if has_tool_call:
        tool_call_examples += 1

role_seq_dist = Counter(tuple(m.get("role") for m in _get_messages(ex)) for ex in dataset)

print("DATA QUALITY CHECK")
print(f"  Total examples: {len(dataset)}")
print(f"  Bad structure: {len(bad_examples)}")
print(f"  Empty final responses: {len(empty_final_responses)}")
print(f"  With explicit tool_calls: {tool_call_examples} ({100*tool_call_examples/len(dataset):.1f}%)")
print(f"  Without explicit tool_calls: {len(dataset)-tool_call_examples} ({100*(len(dataset)-tool_call_examples)/len(dataset):.1f}%)")

print(f"\nTop role sequences (up to 5):")
for seq, n in role_seq_dist.most_common(5):
    print(f"  {seq} -> {n}")

if bad_examples:
    print(f"\n  Bad examples (first 5):")
    for idx, reason in bad_examples[:5]:
        print(f"    Example {idx}: {reason}")

if empty_final_responses:
    print(f"\n  Filtering {len(empty_final_responses)} rows with empty/non-assistant final response...")
    bad_set = set(empty_final_responses)
    good_indices = [i for i in range(len(dataset)) if i not in bad_set]
    dataset = dataset.select(good_indices)
    print(f"  Dataset after filtering: {len(dataset)} examples")

print(f"\nCANCER TYPE DISTRIBUTION:")
max_name_len = max(len(n) for n in cancer_type_counts)
for name, count in sorted(cancer_type_counts.items(), key=lambda x: -x[1]):
    bar = "#" * max(1, count // 200)
    print(f"  {name:<{max_name_len}} {count:>5}  {bar}")
print(f"  {'TOTAL':<{max_name_len}} {sum(cancer_type_counts.values()):>5}")

print(f"\nDataset validated and ready for training")

DATA QUALITY CHECK
  Total examples: 3000
  Bad structure: 0
  Empty final responses: 0
  With explicit tool_calls: 0 (0.0%)
  Without explicit tool_calls: 3000 (100.0%)

Top role sequences (up to 5):
  ('system', 'user', 'assistant') -> 2213
  ('system', 'user', 'assistant', 'user', 'assistant') -> 576
  ('system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant') -> 130
  ('system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant') -> 81

CANCER TYPE DISTRIBUTION:
  multi                    273  #
  pubmed_brain_tumour      273  #
  pubmed_bone_cancer       273  #
  pubmed_colon_cancer      273  #
  pubmed_breast_cancer     273  #
  pubmed_ovarian_cancer    273  #
  pubmed_lung_cancer       273  #
  pubmed_kidney_cancer     273  #
  pubmed_gastric_cancer    272  #
  pubmed_skin_cancer       272  #
  pubmed_prostate_cancer   272  #
  TOTAL                   3000

Dataset validated and ready for training


## 4. Load Model & Tokenizer (4-bit)

In [4]:
import re
import torch

# Disable Unsloth's broken Qwen3.6 compile/flex-attention path before importing it.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
from unsloth import FastLanguageModel

# ── Work around broken optional gptqmodel dependency ─────────────────────────
# The installed gptqmodel cannot import with this Transformers version, but
# PEFT imports its AWQ class merely because the package is installed.
import types

class _AwqGEMMQuantLinearStub:
    pass

_gptqmodel = types.ModuleType("gptqmodel")
_gptqmodel.__path__ = []
_gptqmodel_nn_modules = types.ModuleType("gptqmodel.nn_modules")
_gptqmodel_nn_modules.__path__ = []
_gptqmodel_qlinear = types.ModuleType("gptqmodel.nn_modules.qlinear")
_gptqmodel_qlinear.__path__ = []
_gptqmodel_gemm_awq = types.ModuleType("gptqmodel.nn_modules.qlinear.gemm_awq")
_gptqmodel_gemm_awq.AwqGEMMQuantLinear = _AwqGEMMQuantLinearStub

sys.modules.update({
    "gptqmodel": _gptqmodel,
    "gptqmodel.nn_modules": _gptqmodel_nn_modules,
    "gptqmodel.nn_modules.qlinear": _gptqmodel_qlinear,
    "gptqmodel.nn_modules.qlinear.gemm_awq": _gptqmodel_gemm_awq,
})

# importlib.util.find_spec() raises ValueError on a module whose __spec__ is None.
import importlib.machinery

for _stub_name in (
    "gptqmodel",
    "gptqmodel.nn_modules",
    "gptqmodel.nn_modules.qlinear",
    "gptqmodel.nn_modules.qlinear.gemm_awq",
):
    _stub = sys.modules[_stub_name]
    _stub.__spec__ = importlib.machinery.ModuleSpec(
        _stub_name, loader=None, is_package=hasattr(_stub, "__path__")
    )

# There is no working GPTQModel behind the stub, so PEFT must not take its AWQ path.
def _gptqmodel_unavailable():
    return False

import peft.import_utils as _peft_import_utils

_peft_import_utils.is_gptqmodel_available.cache_clear()
_peft_import_utils.is_gptqmodel_available = _gptqmodel_unavailable
for _mod in list(sys.modules.values()):
    if getattr(_mod, "__name__", "").startswith("peft") and getattr(
        _mod, "is_gptqmodel_available", None
    ) is not None:
        _mod.is_gptqmodel_available = _gptqmodel_unavailable
# ───────────────────────────────────────────────────────────────────────────────

# ── Keep FP8 layers out of the NVFP4 group ───────────────────────────────────
# Transformers removes the FP8 config group before matching, so the NVFP4 group's
# broad mlp target also claims layers 56-63 (stored as FP8) and their
# weight_packed tensors load as MISSING, crashing decompression.
from copy import deepcopy
import compressed_tensors.quantization as _ctq
from transformers.quantizers.quantizer_compressed_tensors import (
    CompressedTensorsHfQuantizer as _CTQuantizer,
    _is_fp8_scheme,
)

if not getattr(_CTQuantizer, "_nvfp4_fp8_target_fix", False):
    _orig_before_load = _CTQuantizer._process_model_before_weight_loading

    def _process_model_before_weight_loading(self, model, **kwargs):
        ct_config = self.compressor.quantization_config
        fp8_targets = [
            target
            for group in ct_config.config_groups.values()
            if _is_fp8_scheme(group)
            for target in group.targets
        ]
        if not (self.use_fp8_kernel and fp8_targets):
            return _orig_before_load(self, model, **kwargs)

        _real_apply = _ctq.apply_quantization_config

        def _apply_with_fp8_excluded(model_, config_, *args, **kwargs_):
            config_ = deepcopy(config_)
            config_.ignore = list(config_.ignore) + [
                t for t in fp8_targets if t not in config_.ignore
            ]
            return _real_apply(model_, config_, *args, **kwargs_)

        _ctq.apply_quantization_config = _apply_with_fp8_excluded
        try:
            return _orig_before_load(self, model, **kwargs)
        finally:
            _ctq.apply_quantization_config = _real_apply

    _CTQuantizer._process_model_before_weight_loading = (
        _process_model_before_weight_loading
    )
    _CTQuantizer._nvfp4_fp8_target_fix = True
# ───────────────────────────────────────────────────────────────────────────────

# The FP8 kernel path is inference-only, so training requires dequantize=True.
from transformers import AutoConfig
from transformers.utils.quantization_config import CompressedTensorsConfig

_base_config = AutoConfig.from_pretrained(BASE_LLM)
_quant_config = CompressedTensorsConfig.from_dict(
    {**_base_config.quantization_config, "dequantize": True}
)

# Load the pre-quantized NVFP4 checkpoint for LoRA training.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    quantization_config=_quant_config,
)

# The weights are dequantized, but the leftover compressed-tensors QDQ forward
# still fake-quantizes activations under torch.no_grad(), severing the autograd graph.
_qdq_disabled = 0
for _module in model.modules():
    if getattr(_module, "quantization_scheme", None) is not None:
        _module.quantization_enabled = False
        _qdq_disabled += 1
print(f"  Disabled compressed-tensors QDQ on {_qdq_disabled} modules (required for training)")

# Qwen3.6-27B is a VLM — Unsloth returns a Qwen3VLProcessor (multimodal wrapper),
# not a raw tokenizer. For text-only SFT, extract the inner tokenizer so that
# chat template / SFTTrainer get a real PreTrainedTokenizer.
if hasattr(tokenizer, "tokenizer"):
    processor = tokenizer
    tokenizer = processor.tokenizer
    print("  (Extracted tokenizer from processor — text-only SFT mode)")

# Fix pad token — set pad = eos for causal LM training.
if tokenizer.pad_token is None or tokenizer.pad_token_id != tokenizer.eos_token_id:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"  Set pad_token = eos_token ({tokenizer.eos_token!r}, id={tokenizer.eos_token_id})")

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id

print(f"Model loaded: {BASE_LLM}")
print(f"  Precision: NVFP4 LoRA")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

==((====))==  Unsloth 2026.7.5: Fast Qwen3_5 patching. Transformers: 5.15.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Compressing model: 100%|██████████| 417/417 [00:01<00:00, 231.09it/s]


Loading weights:   0%|          | 0/1953 [00:00<?, ?it/s]

Decompressing model: 100%|██████████| 417/417 [00:07<00:00, 57.29it/s] 


  Disabled compressed-tensors QDQ on 417 modules (required for training)
  (Extracted tokenizer from processor — text-only SFT mode)
  Set pad_token = eos_token ('<|im_end|>', id=248046)
Model loaded: unsloth/Qwen3.6-27B-NVFP4
  Precision: NVFP4 LoRA
  Max sequence length: 4096
  Vocab size: 248077
  GPU allocated: 56.6 GB


## 5. Format Dataset for Chat Template

Map ShareGPT roles to chat template roles, apply Qwen3's native chat template (ChatML), then manual sequence packing for 100% token utilization.

**Note:** `enable_thinking=False` is used in `apply_chat_template` because the training data already contains `<think>` blocks in the GPT response text. Without this flag, the template would inject a second `<think>` tag.


In [5]:
from datasets import Dataset as HFDataset
import json

# Map standard ShareGPT roles to chat-template roles.
ROLE_MAP = {"system": "system", "human": "user", "gpt": "assistant"}

def _ensure_mapping(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return {}
        try:
            parsed = json.loads(s)
            return parsed if isinstance(parsed, dict) else {"value": parsed}
        except Exception:
            return {"value": value}
    return {"value": value}

def _normalize_tool_fields(messages):
    normalized = []
    for m in messages:
        m2 = dict(m)

        if m2.get("role") == "assistant" and isinstance(m2.get("tool_calls"), list):
            new_tool_calls = []
            for tc in m2["tool_calls"]:
                tc2 = dict(tc)
                fn = dict(tc2.get("function", {}))
                fn["arguments"] = _ensure_mapping(fn.get("arguments", {}))
                tc2["function"] = fn
                new_tool_calls.append(tc2)
            m2["tool_calls"] = new_tool_calls

        normalized.append(m2)
    return normalized

chat_conversations = []
for example in dataset:
    if "messages" in example:
        messages = example["messages"]
    else:
        messages = [
            {"role": ROLE_MAP[turn["from"]], "content": turn["value"]}
            for turn in example["conversations"]
        ]

    messages = _normalize_tool_fields(messages)
    chat_conversations.append(messages)

# enable_thinking=False because data already has <think> blocks in responses.
formatted_texts = tokenizer.apply_chat_template(
    chat_conversations,
    tokenize=False,
    enable_thinking=False,
)

# ── Manual sequence packing ──
eos_id = tokenizer.eos_token_id
num_conversations = 0
all_ids = []

for text in formatted_texts:
    if not text:
        continue
    ids = tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]
    all_ids.extend(ids)
    all_ids.append(eos_id)
    num_conversations += 1

total_tokens = len(all_ids)
num_chunks = total_tokens // MAX_SEQ_LENGTH
all_ids = all_ids[:num_chunks * MAX_SEQ_LENGTH]
chunks = [all_ids[i * MAX_SEQ_LENGTH:(i + 1) * MAX_SEQ_LENGTH] for i in range(num_chunks)]

packed_texts = tokenizer.batch_decode(chunks, skip_special_tokens=False)
dataset = HFDataset.from_dict({"text": packed_texts})
dataset = dataset.shuffle(seed=42)

wasted = total_tokens - (num_chunks * MAX_SEQ_LENGTH)
print(f"Dataset packed: {num_conversations} conversations -> {num_chunks} chunks of {MAX_SEQ_LENGTH} tokens")
print(f"  Total tokens: {total_tokens:,}  |  Wasted (tail): {wasted:,} ({100*wasted/total_tokens:.1f}%)")
print(f"  Token utilization: ~100% (no padding)")
print(f"\n--- Sample packed text (first 500 chars) ---")
print(dataset[0]["text"][:500])


Dataset packed: 3000 conversations -> 1935 chunks of 4096 tokens
  Total tokens: 7,926,142  |  Wasted (tail): 382 (0.0%)
  Token utilization: ~100% (no padding)

--- Sample packed text (first 500 chars) ---
 clinical variables.

Hmm, the user is clearly looking for expert inference beyond the abstract. As an oncologist, I know ovarian cancer prognosis is heavily influenced by molecular factors not captured here. I should consider:
- Homologous recombination deficiency (HRD) status, especially BRCA mutations which are standard biomarkers in ovarian cancer
- TP53 as the near-ubiquitous mutation in high-grade serous carcinoma
- Emerging markers like ARID1A for clear cell carcinoma
- Tumor-infiltrating


## 6. Add LoRA Adapters

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA adapters added (r={LORA_R}, alpha={LORA_ALPHA})")
print(f"  Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.2f}%)")
print(f"  Target modules: {LORA_TARGET_MODULES}")

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
LoRA adapters added (r=32, alpha=32)
  Trainable: 159,383,552 / 28,453,877,856 params (0.56%)
  Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## 7. Trainer Setup

In [7]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=3,
        report_to="none",
        dataset_num_proc=1,
    ),
)

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
num_steps = (len(dataset) * TARGET_EPOCHS + effective_batch_size - 1) // effective_batch_size
print(f"Trainer configured (PubMed Oncology Qwen3.6 27B NVFP4 LoRA — pre-packed)")
print(f"  Effective batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {effective_batch_size}")
print(f"  Epochs: {TARGET_EPOCHS}  |  Steps: ~{num_steps}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Packing: manual (pre-packed, each example = {MAX_SEQ_LENGTH} tokens, zero padding)")
print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")
print(f"  Dataset: {len(dataset)} packed chunks")

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/1935 [00:00<?, ? examples/s]

Trainer configured (PubMed Oncology Qwen3.6 27B NVFP4 LoRA — pre-packed)
  Effective batch size: 2 x 4 = 8
  Epochs: 1  |  Steps: ~242
  LR: 0.0002
  Packing: manual (pre-packed, each example = 4096 tokens, zero padding)
  Precision: bf16
  Dataset: 1935 packed chunks


## 8. Train

In [8]:
from transformers.trainer_utils import get_last_checkpoint

last_ckpt = get_last_checkpoint(trainer.args.output_dir)
if last_ckpt is not None:
    print(f"Resuming from checkpoint: {last_ckpt}")
    result = trainer.train(resume_from_checkpoint=True)
else:
    print("No previous checkpoint found — starting fresh.")
    result = trainer.train()

print(f"\n✓ Training complete!")
print(f"  Final loss:     {result.training_loss:.4f}")
print(f"  Total steps:    {result.global_step}")
print(f"  Training time:  {result.metrics.get('train_runtime', 0) / 60:.1f} minutes")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


No previous checkpoint found — starting fresh.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,935 | Num Epochs = 1 | Total steps = 242
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 159,383,552 of 28,453,877,856 (0.56% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.053992
2,1.202925
3,1.160159
4,1.067173
5,0.959499
6,0.953713
7,0.945764
8,0.907900
9,0.858638
10,0.917235


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_nvfp4/train/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_nvfp4/train/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_nvfp4/train/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_nvfp4/train/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_nvfp4/train/checkpoint-242/tokenizer_config.json.



✓ Training complete!
  Final loss:     0.6817
  Total steps:    242
  Training time:  542.6 minutes


## 9. Save LoRA Adapters

Save the trained LoRA adapters and system prompts. The DPO notebook (Phase 2) expects the LoRA at this path.

In [9]:
import json
from pathlib import Path

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

# Save system prompts alongside adapters for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/oncologist_system_prompts.json"
system_prompts_by_cancer=""
with open(prompts_path, "w") as f:
    json.dump(system_prompts_by_cancer, f, indent=2)

print(f"\nLoRA adapters saved!")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompts: {prompts_path} ({len(system_prompts_by_cancer)} cancer types)")
print(f"  DPO notebook expects LoRA at: {{PROJECT_ROOT}}/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_nvfp4/lora_adapters")

for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

Saving LoRA adapters to /workspace/training/pubmed/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_nvfp4/lora_adapters...


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_nvfp4/lora_adapters/tokenizer_config.json.



LoRA adapters saved!
  Adapters:       /workspace/training/pubmed/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_nvfp4/lora_adapters
  System prompts: /workspace/training/pubmed/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_nvfp4/lora_adapters/oncologist_system_prompts.json (0 cancer types)
  DPO notebook expects LoRA at: {PROJECT_ROOT}/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_nvfp4/lora_adapters
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                   608.1 MB
  chat_template.jinja                           0.0 MB
  oncologist_system_prompts.json                0.0 MB
  tokenizer.json                               19.1 MB
  tokenizer_config.json                         0.0 MB


## 10. Test Inference

Smoke test with oncology questions across different cancer types. This is a thinking model — responses will include `<think>` reasoning blocks.

In [12]:
import warnings, logging, json
from transformers import TextStreamer

# Suppress transformers deprecation warning bug (msg % FutureWarning formatting error)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("transformers.modeling_attn_mask_utils").setLevel(logging.ERROR)

# Load model from saved adapters if not already in memory
if "model" not in dir() or model is None:
    from unsloth import FastLanguageModel
    print(f"Model not in memory — loading from saved adapters: {LORA_OUTPUT_DIR}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=LORA_OUTPUT_DIR,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,
    )
    # Qwen3.6-27B VLM — extract inner tokenizer from processor wrapper
    if hasattr(tokenizer, "tokenizer"):
        tokenizer = tokenizer.tokenizer
        
if "system_prompts_by_cancer" not in dir() or not system_prompts_by_cancer:
    with open(f"{LORA_OUTPUT_DIR}/oncologist_system_prompts.json") as f:
        system_prompts_by_cancer = json.load(f)

FastLanguageModel.for_inference(model)

test_cancers = list(system_prompts_by_cancer.keys())[:3]

print(f"INFERENCE TEST — {len(test_cancers)} CANCER TYPES\n")

for cancer_type in test_cancers:
    system_prompt = system_prompts_by_cancer[cancer_type]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    # Thinking model: let Qwen3 default (enable_thinking=True) inject <think>
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    print(f"{'='*60}")
    print(f"  CANCER TYPE: {cancer_type.upper()}")
    print(f"  Q: {TEST_PROMPT}")
    print(f"  A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs

NameError: name 'null' is not defined

## 11. Verify Adapter (Cold Reload)

Loads adapters from disk in a fresh model to confirm portability. Also verifies the DPO notebook can load these adapters.

In [ ]:
import gc, torch, json
from pathlib import Path

del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

from unsloth import FastLanguageModel

model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
)
# Qwen3.6-27B VLM — extract inner tokenizer from processor wrapper
if hasattr(tokenizer2, "tokenizer"):
    tokenizer2 = tokenizer2.tokenizer

FastLanguageModel.for_inference(model2)

with open(f"{LORA_OUTPUT_DIR}/oncologist_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

test_cancer = list(reloaded_prompts.keys())[0]
test_system = reloaded_prompts[test_cancer]

messages = [
    {"role": "system", "content": test_system},
    {"role": "user", "content": TEST_PROMPT},
]

text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer2(text, return_tensors="pt").to(model2.device)

outputs = model2.generate(
    **inputs,
    max_new_tokens=2048,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    do_sample=True,
)

response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (cancer type: {test_cancer}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:500]}")
print(f"\nAdapter loads cleanly from disk.")
print(f"DPO notebook (Phase 2) can now load this LoRA from: {LORA_OUTPUT_DIR}")

print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()